gold_agg: Gold post_counts_by_user (Iceberg + MERGE)

Tarea 3 del DAG. Este notebook es el respaldo manual si Airflow falla.

Cruza silver.posts_hist con silver.users_hist y genera una fila por usuario con su
actividad (preguntas, respuestas, puntuación, respuestas aceptadas, tendencia entre años) y su perfil

Escribe con MERGE en nessie.gold.post_counts_by_user, con la columna load_date

Reglas de negocio:
Solo cuentan preguntas (tipo 1) y respuestas (tipo 2); las fichas de etiquetas (tipos 4 y 5) se excluyen.
Se excluyen los posts sin autor. Todo lo excluido queda contado en `silver.dq_log`
activity_tier: `alto` / `medio` / `bajo` según el total de posts (umbrales en la celda 2)
activity_trend: `creciente` / `estable` / `decreciente` según los posts del segundo año frente al primero.
Las respuestas aceptadas solo se ven si la pregunta y la respuesta caen en los periodos cargados.

Requiere: `posts_hist` y `users_hist` en Silver

Idempotente: si nada cambió, no escribe nada.

In [1]:
import os
import sys

# La carpeta jobs/ esta montada en Jupyter (/workspace/jobs) y en Airflow (/opt/airflow/jobs)
JOBS = os.environ.get("JOBS_DIR") or next(p for p in ("/workspace/jobs", "/opt/airflow/jobs") if os.path.isdir(p))
sys.path.insert(0, JOBS)

from pyspark import StorageLevel
from pyspark.sql import functions as F

import config
import silver_lib

# Parametros de negocio
TIER_ALTO = 10                # posts totales desde los que un usuario es alto  
TIER_MEDIO = 3                # posts totales desde los que es medio
YEAR_1, YEAR_2 = 2020, 2021   # anios que se comparan para la tendencia

POSTS = silver_lib.table_name(config.SILVER_NS, "posts_hist")
USERS = silver_lib.table_name(config.SILVER_NS, "users_hist")
TABLE = silver_lib.table_name(config.GOLD_NS, "post_counts_by_user")
print("Origen:", POSTS, "+", USERS, "| destino:", TABLE)

Origen: nessie.silver.posts_hist + nessie.silver.users_hist | destino: nessie.gold.post_counts_by_user


In [2]:
spark = silver_lib.get_spark("gold_agg")
silver_lib.ensure_namespaces(spark)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
  user_id BIGINT, display_name STRING, location STRING, reputation BIGINT,
  account_created TIMESTAMP, last_access_date TIMESTAMP,
  questions BIGINT, answers BIGINT, total_posts BIGINT, posts_{YEAR_1} BIGINT, posts_{YEAR_2} BIGINT,
  total_score BIGINT, avg_score_per_post DOUBLE, accepted_answers BIGINT, acceptance_rate DOUBLE,
  first_post_date TIMESTAMP, last_post_date TIMESTAMP, answers_per_question DOUBLE,
  activity_tier STRING, activity_trend STRING, row_hash STRING, load_date TIMESTAMP
) USING iceberg
""")
print("Tabla lista:", TABLE)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Tabla lista: nessie.gold.post_counts_by_user


In [3]:
# Cruce y agregacion: una fila por usuario con actividad (preguntas y respuestas), enriquecida con su perfil.
# Reglas: solo post_type_id 1 (pregunta) y 2 (respuesta); se excluyen posts sin autor, LEFT JOIN a users_hist
# para no perder autores cuyo perfil no esta en Silver. 
# Respuestas aceptadas: solo las visibles en los periodos cargados.

metrics = spark.sql(f"""
WITH elegibles AS (
  SELECT post_id, post_type_id, owner_user_id, creation_date, score
  FROM {POSTS}
  WHERE post_type_id IN (1, 2) AND owner_user_id IS NOT NULL
),
aceptadas AS (
  SELECT DISTINCT accepted_answer_id AS post_id
  FROM {POSTS}
  WHERE post_type_id = 1 AND accepted_answer_id IS NOT NULL
),
por_usuario AS (
  SELECT
    e.owner_user_id AS user_id,
    SUM(CASE WHEN e.post_type_id = 1 THEN 1 ELSE 0 END) AS questions,
    SUM(CASE WHEN e.post_type_id = 2 THEN 1 ELSE 0 END) AS answers,
    COUNT(*) AS total_posts,
    SUM(CASE WHEN year(e.creation_date) = {YEAR_1} THEN 1 ELSE 0 END) AS posts_{YEAR_1},
    SUM(CASE WHEN year(e.creation_date) = {YEAR_2} THEN 1 ELSE 0 END) AS posts_{YEAR_2},
    SUM(e.score) AS total_score,
    SUM(CASE WHEN e.post_type_id = 2 AND a.post_id IS NOT NULL THEN 1 ELSE 0 END) AS accepted_answers,
    MIN(e.creation_date) AS first_post_date,
    MAX(e.creation_date) AS last_post_date
  FROM elegibles e
  LEFT JOIN aceptadas a ON e.post_id = a.post_id
  GROUP BY e.owner_user_id
)
SELECT
  p.user_id, u.display_name, u.location, u.reputation,
  u.creation_date AS account_created, u.last_access_date,
  p.questions, p.answers, p.total_posts, p.posts_{YEAR_1}, p.posts_{YEAR_2},
  p.total_score,
  ROUND(p.total_score / p.total_posts, 2) AS avg_score_per_post,
  p.accepted_answers,
  CASE WHEN p.answers > 0 THEN ROUND(p.accepted_answers / p.answers, 4) END AS acceptance_rate,
  p.first_post_date, p.last_post_date,
  CASE WHEN p.questions > 0 THEN ROUND(p.answers / p.questions, 2) END AS answers_per_question,
  CASE WHEN p.total_posts >= {TIER_ALTO} THEN 'alto'
       WHEN p.total_posts >= {TIER_MEDIO} THEN 'medio'
       ELSE 'bajo' END AS activity_tier,
  CASE WHEN p.posts_{YEAR_2} > p.posts_{YEAR_1} THEN 'creciente'
       WHEN p.posts_{YEAR_2} < p.posts_{YEAR_1} THEN 'decreciente'
       ELSE 'estable' END AS activity_trend
FROM por_usuario p
LEFT JOIN {USERS} u ON p.user_id = u.user_id
""").persist(StorageLevel.DISK_ONLY)  # se calcula una vez; a disco para no usar memoria

print("Usuarios con actividad:", metrics.count())
metrics.orderBy(F.col("total_posts").desc()).select(
    "user_id", "display_name", "questions", "answers", "total_posts", "activity_tier", "activity_trend"
).show(5, truncate=False)

Usuarios con actividad: 1098420
+--------+-------------+---------+-------+-----------+-------------+--------------+
|user_id |display_name |questions|answers|total_posts|activity_tier|activity_trend|
+--------+-------------+---------+-------+-----------+-------------+--------------+
|1144035 |Gordon Linoff|0        |13332  |13332      |alto         |decreciente   |
|3732271 |akrun        |0        |7283   |7283       |alto         |decreciente   |
|3962914 |Ronak Shah   |1        |6569   |6570       |alto         |creciente     |
|2901002 |jezrael      |4        |5100   |5104       |alto         |decreciente   |
|10676716|GMB          |0        |3921   |3921       |alto         |decreciente   |
+--------+-------------+---------+-------+-----------+-------------+--------------+
only showing top 5 rows



In [4]:
# Controles de integridad: lo que se excluye queda contado y la suma de posts por usuario debe cuadrar
p = spark.sql(f"""
SELECT COUNT(*) AS total,
  SUM(CASE WHEN post_type_id IN (1, 2) AND owner_user_id IS NOT NULL THEN 1 ELSE 0 END) AS elegibles,
  SUM(CASE WHEN post_type_id IN (1, 2) AND owner_user_id IS NULL THEN 1 ELSE 0 END) AS sin_autor,
  SUM(CASE WHEN post_type_id NOT IN (1, 2) THEN 1 ELSE 0 END) AS otros_tipos
FROM {POSTS}
""").first()
m = metrics.agg(
    F.count(F.lit(1)).alias("usuarios"),
    F.sum("total_posts").alias("suma_posts"),
    F.sum(F.when(F.col("account_created").isNull(), 1).otherwise(0)).alias("sin_perfil"),
).first()
n = lambda x: int(x or 0)

checks = [
    ("posts_sin_autor_excluidos", n(p["sin_autor"]), n(p["total"]), "excluida"),
    ("posts_otros_tipos_excluidos", n(p["otros_tipos"]), n(p["total"]), "excluida"),
    ("usuarios_sin_perfil", n(m["sin_perfil"]), n(m["usuarios"]), "informativa (perfil NULL)"),
    ("descuadre_total_posts", abs(n(m["suma_posts"]) - n(p["elegibles"])), n(p["elegibles"]), "control (debe ser 0)"),
]
for regla, filas, total, accion in checks:
    print(f"{regla:<28} {filas:>10} / {total:<10} {accion}")

posts_sin_autor_excluidos         63032 / 4170061    excluida
posts_otros_tipos_excluidos        3856 / 4170061    excluida
usuarios_sin_perfil                   1 / 1098420    informativa (perfil NULL)
descuadre_total_posts                 0 / 4103173    control (debe ser 0)


In [5]:
VALUE_COLS = [  # columnas que entran en el hash de cambios
    "display_name", "location", "reputation", "account_created", "last_access_date",
    "questions", "answers", "total_posts", f"posts_{YEAR_1}", f"posts_{YEAR_2}", "total_score",
    "avg_score_per_post", "accepted_answers", "acceptance_rate", "first_post_date", "last_post_date",
    "answers_per_question", "activity_tier", "activity_trend",
]
OUTPUT_COLS = ["user_id"] + VALUE_COLS + ["row_hash", "load_date"]


def with_hash_and_load_date(df):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in VALUE_COLS]
    return (df.withColumn("row_hash", F.sha2(F.concat_ws("||", *parts), 256))
              .withColumn("load_date", F.current_timestamp())
              .select(*OUTPUT_COLS))


def changed_rows(df, target):
    """Solo lo NUEVO o MODIFICADO: filas cuyo (user_id, row_hash) no esta en el destino."""
    out = df.join(target.select("user_id", "row_hash"), ["user_id", "row_hash"], "left_anti")
    return out.select(*df.columns)  # el join reordena columnas: se restaura el orden de la tabla


delta = changed_rows(with_hash_and_load_date(metrics), spark.table(TABLE)).persist(StorageLevel.DISK_ONLY)
n_delta = delta.count()
print(f"Filas nuevas o modificadas: {n_delta}")

Filas nuevas o modificadas: 0


In [6]:
if n_delta:
    delta.createOrReplaceTempView("gold_src")
    spark.sql(f"""
        MERGE INTO {TABLE} t USING gold_src s ON t.user_id = s.user_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("MERGE aplicado")
else:
    print("Sin cambios: no se escribe nada en la tabla")
delta.unpersist()
metrics.unpersist()
silver_lib.log_dq(spark, TABLE, checks)

Sin cambios: no se escribe nada en la tabla


  DQ nessie.gold.post_counts_by_user posts_sin_autor_excluidos      63032 / 4170061    excluida
  DQ nessie.gold.post_counts_by_user posts_otros_tipos_excluidos       3856 / 4170061    excluida
  DQ nessie.gold.post_counts_by_user usuarios_sin_perfil               1 / 1098420    informativa (perfil NULL)
  DQ nessie.gold.post_counts_by_user descuadre_total_posts             0 / 4103173    control (debe ser 0)


In [7]:
t = spark.table(TABLE)
n_rows = t.count()
n_ids = t.select("user_id").distinct().count()
print(f"Filas: {n_rows} | user_id distintos: {n_ids} (deben ser iguales)")
spark.sql(f"SELECT activity_tier, COUNT(*) AS usuarios FROM {TABLE} GROUP BY activity_tier ORDER BY usuarios DESC").show()
spark.sql(f"SELECT activity_trend, COUNT(*) AS usuarios FROM {TABLE} GROUP BY activity_trend ORDER BY usuarios DESC").show()
spark.sql(f"SELECT count(*) AS archivos_datos FROM {TABLE}.files").show()
spark.sql(f"SELECT count(*) AS snapshots FROM {TABLE}.snapshots").show()
t.orderBy(F.col("total_posts").desc()).select(
    "user_id", "display_name", "reputation", "total_posts", "accepted_answers", "acceptance_rate",
    "activity_tier", "load_date").show(5, truncate=False)

Filas: 1098420 | user_id distintos: 1098420 (deben ser iguales)
+-------------+--------+
|activity_tier|usuarios|
+-------------+--------+
|         bajo|  794750|
|        medio|  240307|
|         alto|   63363|
+-------------+--------+

+--------------+--------+
|activity_trend|usuarios|
+--------------+--------+
|   decreciente|  573751|
|     creciente|  482150|
|       estable|   42519|
+--------------+--------+

+--------------+
|archivos_datos|
+--------------+
|             8|
+--------------+

+---------+
|snapshots|
+---------+
|        1|
+---------+



+--------+-------------+----------+-----------+----------------+---------------+-------------+--------------------------+
|user_id |display_name |reputation|total_posts|accepted_answers|acceptance_rate|activity_tier|load_date                 |
+--------+-------------+----------+-----------+----------------+---------------+-------------+--------------------------+
|1144035 |Gordon Linoff|1255823   |13332      |5921            |0.4441         |alto         |2026-10-11 18:03:37.499274|
|3732271 |akrun        |879355    |7283       |4027            |0.5529         |alto         |2026-10-11 18:03:37.499274|
|3962914 |Ronak Shah   |383528    |6570       |3998            |0.6086         |alto         |2026-10-11 18:03:37.499274|
|2901002 |jezrael      |845804    |5104       |3934            |0.7714         |alto         |2026-10-11 18:03:37.499274|
|10676716|GMB          |218936    |3921       |2128            |0.5427         |alto         |2026-10-11 18:03:37.499274|
+--------+-------------+

In [8]:
spark.stop()